# Stochastic Linear Bandits with context - LinUCB - A completely toy scenario 

### ***Summer School - June, 29 2026***

A demand response aggregator (DRA) aims to sell load reductions (and recently upward variations; but we let this case for another summer school) on short-term electricity markets (see [NEBCO](https://www.services-rte.com/en/learn-more-about-our-services/participate-nebef-mechanism) mechanism explanations provided by RTE for further details).

We assume it can ask intermittent load reductions to various assets:
- Electrical vehicules (EV) that switch on for charging during the night (8 pm - 7am) or mid-day (10am - 5pm) with a mean hourly load of $E_{\mathrm{EV}}$kWh (domestic power socket);
- Hot water tanks that switch on almost at random throughout the day (mainly when the hot water inside cools down to a set temperature or when hot water is drawn) with a mean hourly load of $E_{\mathrm{HW}}$kWh;
- Electric heaters that switch on when the outside temperature is below 15°C with a mean hourly load of  $E_{\mathrm{HT}} \times \max(0, 15° - T_t)$ kWh;
- Air conditionings that switch on when the outside temperature exceed 25°C with a mean hourly load of 2 $E_{\mathrm{AC}}\times \max(0, T_t - 25°)$ kWh.

Some devices can be directly controlled using hardware installed in customers' homes. For heatings and ACs, the DRA will interrupt usage for one half-hour. The customer can of course turn their heating or AC back on. For the hot water tanks, it is similar, except that it automatically turns back on if the water becomes too cold (if customer take showers during the half-hour in question, e.g.). Finally, for the EVs, let’s assume that the DRA sends a signal to the consumer (a text message), who can choose whether or not to delay charging by one half-hour. 

In order to be able to make a credible (and profitable) offer on the market, the DRA must accurately estimate the reduction it is offering, whilst seeking to maximise it (we entrust the pricing strategy to it). 

Let’s assume the DRA has access to large numbers of electrical appliances (a large number of EVs, a large number of hot-water tanks, etc.). Furthermore, we assume that at any instant $t$, at least $N$ hot water tanks are switched on, as are EVs during charging periods and air conditioning and heating systems when temperatures are very hot or cold. Thus, the DRA can propose truning off the $N$ active devices of the population it has chosen. The chosen population is denoted $a_t$. Then, it observes the average load reduction $Y_t$ (in pratice, this is done by RTE). 

To model the variability in the customer responses (of the $N$ devices in question, some do not switch off, others consume more power than expected, etc.), we assume this average load reduction to be perturbed by a random gaussian noise $\eta_t$. If $N$ is large enough, this model holds up well thanks to the law of large numbers; with the exception that the variance would depend on the chosen population $A_t$.

We assume that the DRA does not know the mean load of each device. However, it does know the charging time windows of EVs and that heating and AC loads are proportional to temperature differences.

Let's write the problem! We introduce
 - the action set: $[K],$ with $K=4$ and $1$ of EVs, $2$ for heating etc.;
 - the mapping context function to model EVs availability (only during charging hours) and temperature imapct of heatings and ACs. For any half-hour $hh$, temperature $T$ and device $a\in [K]$, it return the 4-dimensional vector:

  $$ \phi: (\mathrm{hh},T,a) \mapsto \begin{pmatrix} \mathbf{1}_{hh_t \in [\mathrm{charging hours}]} \times \mathbf{1}_{a=1}\\ \mathbf{1}_{a=2} \\ \max(0, 15 - T) \times \mathbf{1}_{a=3} \\ \max(0, T - 25)\times \mathbf{1}_{a=4} \\ \end{pmatrix} \,;$$

  - and the parameter: $\theta = (E_{\mathrm{EV}},E_{\mathrm{HW}},E_{\mathrm{HT}},E_{\mathrm{AC}})^\mathrm{T}$ (mean half-hourly load of each device). 

At an instant $t$, the DRA observes $x_t = (hh_t,T_t)$ and if it picks action $a$, the average reduction is $$Y_t =  \langle \theta , \, \phi(hh_t,T_t,a) \rangle + \eta_t \quad \mathrm{where} \quad \eta_t \sim \mathcal{N}(0,\sigma^2)\,.$$

We will propose that it uses the LinUCB bandit algorithm to maximize the potential load reduction at each half-hour interval. This algorithm provides an estimate of the expected reward, which can then be used to build offers on the market. However, we recommend that the DRA wait until the algorithm has learned sufficiently before doing so (as RTE evaluates the reliability of DRAs).

In [ ]:
import numpy as np
import pandas as pd 
import random
import matplotlib.pyplot as plt 
from ipywidgets import IntSlider
from IPython.display import display, clear_output

# we use the contextual data we will detail in the last notebook. It contains date_time and 
# temperature (in UK) for the full year 2013
# we also introduce the variable 'half_hour' taking values from 0 (00:00) to 47 (23:30)
# we smooth temperature data to model heating and AC systems inertia

df_context = pd.read_csv('electrical_demand_data.csv', index_col = 'date_time')[['temperature','half_hour']]
df_context.index = pd.to_datetime(df_context.index, format='mixed')
def exp_smooth(vec, alpha):
        if alpha > 1 or alpha < 0:
            return 'alpha doit etre compris entre 0 et 1'
        else:
            smth = np.full(len(vec), np.nan)
            if len(vec) > 0:
                smth[0] = vec[0]
                if len(vec) > 1:
                    for i in range(1, len(vec)):
                        smth[i] = alpha * smth[i - 1] + (1 - alpha) * vec[i]
            return smth

df_context['temperature'] = exp_smooth(df_context['temperature'].to_numpy(), alpha= 0.95) 
df_context.tail(10)

In [ ]:
# Definie parameter and mapping function 

theta  = np.array([8,6,3,2])/2 # half-hourly load of each device 
sigma = 1 

def mapping(context, a):
    heating = max(0,15-context['temperature'])
    ac = max(0, context['temperature'] - 23)
    ev_ischarging = ((context['half_hour']<14) or 
                     (context['half_hour']>=20)*(context['half_hour']<34) or 
                     (context['half_hour']>40))*1
    return np.array([ev_ischarging*(a==1),1*(a==2),heating*(a==3),ac*(a==4)])

In [ ]:
# LinUCB implementaion 

def linucb(df_context, theta, sigma, beta, lamb = 1):

    T = len(df_context)
    d = len(theta)
    K = d
  
    actions = np.zeros(T)
    best_actions = np.zeros(T)

    rewards = np.zeros(T)
    bests = np.zeros(T)

    thetas = np.zeros([T,K])
    
    phi_all = np.array([mapping(df_context.iloc[0, :], a+1) for a in range(K)])
    best_actions[0] = np.argmax(phi_all@theta)+1

    # random choice for first iteration 
    actions[0] = random.randint(1,K)
    phi = mapping(df_context.iloc[0,:], actions[0])

    rewards[0] = phi@theta + random.normalvariate(0, sigma)
    bests[0] = mapping(df_context.iloc[0,:], best_actions[0])@theta

    V = lamb*np.diag(np.ones(d)) + phi.reshape(-1, 1) @ phi.reshape(-1, 1).T
    S = rewards[0] * phi 
    for t in range(1,T):
        solve_V = np.linalg.inv(V)
        hat_theta = solve_V@ S
        thetas[t,:] = hat_theta
        phi_all = np.array([mapping(df_context.iloc[t, :], a+1) for a in range(K)])
        # confidence bonus 
        eps_all = np.sqrt(beta[t]*np.array([(phi_all[i,:].reshape(-1, 1).T@solve_V@phi_all[i,:].reshape(-1, 1))[0][0] for i in range(K)]))
        ucb_all = phi_all@hat_theta + eps_all
        best_actions[t] = np.argmax(phi_all@theta) +1
        actions[t] = np.argmax(ucb_all)+1
        phi = mapping(df_context.iloc[t,:], actions[t])
        rewards[t] = phi@theta + random.normalvariate()
        bests[t] = mapping(df_context.iloc[t,:], best_actions[t])@theta

        V = V + phi.reshape(-1, 1) @ phi.reshape(-1, 1).T
        S = S + rewards[t] * phi 
    return actions, rewards,best_actions, bests, thetas,  hat_theta, S, V

lam = 1
beta = 100*np.log(range(1,len(df_context)+1))/lam

actions, rewards,best_actions, bests, thetas,  hat_theta, S, V = linucb(df_context, theta, sigma, beta, lamb = lam)

In [ ]:
df = df_context.copy()
df['action'] = actions
df['best_actions'] = best_actions
df['reward'] = rewards
df['best_expected_rewards'] = bests
df['theta_1'] = thetas[:,0]
df['theta_2'] = thetas[:,1]
df['theta_3'] = thetas[:,2]
df['theta_4'] = thetas[:,3]


In [ ]:
df = df.copy()
df.index = pd.to_datetime(df.index)

# Half-hour where EVs are available 
df["stripe"] = (
    (df["half_hour"] < 14) |
    ((df["half_hour"] >= 20) & (df["half_hour"] < 34)) |
    (df["half_hour"] > 40)
).astype(int)

action_colors = {
    1: ("purple", "EV"),
    2: ("gold", "Hot Water Tank"),
    3: ("red", "Heating"),
    4: ("blue", "AC"),
}

# create weekly plots
weeks = []
current = df.index.min()
end = df.index.max()

while current < end:
    week_df = df[(df.index >= current) & (df.index < current + pd.Timedelta(days=7))]
    if not week_df.empty:
        weeks.append(week_df)
    current += pd.Timedelta(days=7)

temp_min, temp_max = df["temperature"].min(), df["temperature"].max()
rew_min = min(df["reward"].min(), df["best_expected_rewards"].min())
rew_max = max(df["reward"].max(), df["best_expected_rewards"].max())

def plot_week(week_idx):
    clear_output(wait=True) 

    week_df = weeks[week_idx]

    fig, ax1 = plt.subplots(figsize=(12,6))
    ax2 = ax1.twinx()

    # plot temperature 
    ax1.plot(week_df.index, week_df["temperature"], color="red", label="Temperature",linestyle="--",)

    # reward 
    for action, (color, label) in action_colors.items():
        subset = week_df[week_df["action"] == action]
        if not subset.empty:
            ax2.scatter(subset.index, subset["reward"], color=color, s=10, label=label)

    # and expected best reward
    ax2.plot(
        week_df.index,
        week_df["best_expected_rewards"],
        color="black",
        label="Best Expected Reward"
    )

    for j in range(len(week_df)-1):
        if week_df["stripe"].iloc[j] == 1:
            ax1.axvspan(
                week_df.index[j],
                week_df.index[j+1],
                color="gray",
                alpha=0.1
            )

    # % optimal
    optimal = (week_df["action"] == week_df["best_actions"]).mean() * 100

    # Fixed scales
    ax1.set_ylim(temp_min, temp_max)
    ax2.set_ylim(rew_min, rew_max)

    ax1.set_title(
        f"Week {week_idx+1} | {week_df.index.min().date()} → {week_df.index.max().date()} "
        f"| Best actions picked: {optimal:.1f}%"
    )

    ax1.set_ylabel("Temperature")
    ax2.set_ylabel("Reward")

    # Legend
    handles1, labels1 = ax1.get_legend_handles_labels()
    handles2, labels2 = ax2.get_legend_handles_labels()
    by_label = dict(zip(labels1 + labels2, handles1 + handles2))
    ax1.legend(by_label.values(), by_label.keys(), loc="upper left")

    display(slider) 
    plt.show()


# --- Slider ---
slider = IntSlider(
    min=0,
    max=len(weeks)-1,
    step=1,
    value=0,
    description="Week",
    continuous_update=False   
)

def on_value_change(change):
    if change["name"] == "value":
        plot_week(change["new"])

slider.observe(on_value_change)

# Initial draw
plot_week(0)

In [ ]:
plt.plot(np.cumsum(bests-rewards))
plt.ylabel('Cumulative regret')